# PlanIA P2 · API de generación y descripción

SDXL genera una imagen a partir de una foto y un prompt en `cuda:0`. Qwen3-VL describe el resultado en español desde `cuda:1`.

En Kaggle, activa **Internet**, selecciona **GPU T4 ×2**, crea el secreto **NGROK_AUTHTOKEN** con el authtoken del agente de ngrok y habilita el acceso de este notebook al secreto. Ejecuta todas las celdas. La URL pública aparece en el log y funciona mientras la sesión siga activa. Agrega `/docs` para abrir el formulario de Swagger. El túnel es público y el endpoint acepta solicitudes de quien tenga la URL.

## 1. Instalar dependencias


In [ ]:
%pip install -q diffusers==0.35.2 transformers==4.57.1 accelerate==1.11.0 fastapi uvicorn python-multipart pyngrok

## 2. Leer el secreto y seleccionar las GPUs

Validamos la configuración antes de descargar los pesos.

In [ ]:
import os
import torch
from kaggle_secrets import UserSecretsClient
from PIL import Image, ImageOps
from diffusers import StableDiffusionXLImg2ImgPipeline
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

ngrok_token = os.getenv("NGROK_AUTHTOKEN")
if not ngrok_token:
    try:
        ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    except Exception:
        raise RuntimeError("Agrega NGROK_AUTHTOKEN en Kaggle Secrets y habilítalo para este notebook.") from None

if torch.cuda.device_count() < 2:
    raise RuntimeError("Selecciona GPU T4 ×2 en Accelerator.")
device = torch.device("cuda:0")
qwen_device = torch.device("cuda:1")

## 3. Cargar SDXL en la primera T4

Usamos la revisión y el formato FP16 de la v3.

In [ ]:
model_id = "stabilityai/stable-diffusion-xl-base-1.0"
model_revision = "462165984030d82259a11f4367a4eed129e94a7b"

pipeline = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    model_id,
    revision=model_revision,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipeline.to(device)
pipeline.set_progress_bar_config(disable=True)

## 4. Cargar Qwen3-VL en la segunda T4

El modelo visual se mantiene en `cuda:1` durante toda la sesión.

In [ ]:
qwen_id = "Qwen/Qwen3-VL-4B-Instruct"
qwen_revision = "ebb281ec70b05090aa6165b016eac8ec08e71b17"
processor = AutoProcessor.from_pretrained(qwen_id, revision=qwen_revision)
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    qwen_id,
    revision=qwen_revision,
    dtype=torch.float16,
    device_map={"": qwen_device.index},
    attn_implementation="sdpa",
).eval()

## 5. Definir la API

`POST /generate` recibe los campos `image` y `prompt` como formulario multipart. Devuelve un JSON con `image_base64` (PNG), `image_mime_type` y `description`. El límite de entrada es 10 MiB y la API procesa una generación a la vez. `GET /health` indica que el servidor está listo.


In [ ]:
import base64
from io import BytesIO
from threading import Lock

from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from PIL import UnidentifiedImageError

app = FastAPI(title="PlanIA P2 · SDXL y Qwen")
inference_lock = Lock()
max_upload_bytes = 10 * 1024 * 1024

@app.get("/health")
def health():
    return {"status": "ready"}

@app.post("/generate")
def generate(image: UploadFile = File(...), prompt: str = Form(...)):
    prompt = prompt.strip()
    if not prompt or len(prompt) > 2000:
        raise HTTPException(422, "El prompt debe tener entre 1 y 2000 caracteres.")
    data = image.file.read(max_upload_bytes + 1)
    if len(data) > max_upload_bytes:
        raise HTTPException(413, "La imagen supera 10 MiB.")
    try:
        with Image.open(BytesIO(data)) as source:
            if source.width * source.height > 25_000_000:
                raise HTTPException(413, "La imagen supera 25 megapíxeles.")
            reference = ImageOps.exif_transpose(source).convert("RGB")
    except (UnidentifiedImageError, OSError, Image.DecompressionBombError):
        raise HTTPException(422, "No se pudo leer la imagen.") from None

    if not inference_lock.acquire(blocking=False):
        raise HTTPException(429, "Hay una generación en curso. Intenta nuevamente.")
    try:
        size = (768, 1024) if reference.height > reference.width else (1024, 768)
        prepared = ImageOps.pad(reference, size, Image.Resampling.LANCZOS)
        with torch.inference_mode():
            result = pipeline(
                prompt=prompt, image=prepared, strength=0.50,
                num_inference_steps=25, guidance_scale=7.5,
            ).images[0]
            messages = [{"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": (
                    "Describe en español esta imagen como especialista en fotografía "
                    "y diseño visual. Explica la composición, la iluminación, la "
                    "paleta y las texturas. Menciona solo detalles visibles."
                )},
            ]}]
            chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            preview = ImageOps.contain(result, (768, 768), Image.Resampling.LANCZOS)
            inputs = processor(text=[chat], images=[preview], return_tensors="pt")
            inputs.pop("token_type_ids", None)
            inputs = inputs.to(qwen_device, dtype=torch.float16)
            output = qwen_model.generate(**inputs, max_new_tokens=320, do_sample=False)
            tokens = output[:, inputs["input_ids"].shape[1]:]
            description = processor.batch_decode(tokens, skip_special_tokens=True)[0].strip()
        if not description:
            raise HTTPException(502, "Qwen devolvió una descripción vacía.")
        buffer = BytesIO()
        result.save(buffer, format="PNG")
        return {
            "image_base64": base64.b64encode(buffer.getvalue()).decode("ascii"),
            "image_mime_type": "image/png",
            "description": description,
        }
    finally:
        inference_lock.release()

## 6. Abrir el túnel y mantener el servidor activo

El agente de ngrok usa el authtoken de tu cuenta. Esta celda permanece en ejecución para mantener el puerto 8000 disponible. Al detener la sesión, el enlace deja de responder.


In [ ]:
from contextlib import asynccontextmanager

import uvicorn
from pyngrok import ngrok

@asynccontextmanager
async def tunnel_lifespan(app):
    ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(addr="127.0.0.1:8000", proto="http")
    print(f"API pública: {tunnel.public_url}", flush=True)
    print(f"Formulario: {tunnel.public_url}/docs", flush=True)
    try:
        yield
    finally:
        ngrok.disconnect(tunnel.public_url)
        ngrok.kill()

app.router.lifespan_context = tunnel_lifespan

# Uvicorn comparte el bucle de eventos del notebook.
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000))
await server.serve()